In [5]:
import glob

import polars as pl

In [6]:
# On main computer /mnt/Fast2T/data/link/*.parquet
DATA_DIR = "/Users/qr/Downloads/v5_chunks/*.parquet"

### Most used english stem

In [32]:
some_file = sorted(glob.glob(DATA_DIR))[0]

(pl
 .read_parquet(some_file)
 # only HTML files (enum;2=html)
 .filter(pl.col("type") == 2)
 # temporary for validation
 .filter(pl.col("lang") == "ENGLISH")
 # only important rows
 .select(pl.col("text").str.split(' '))
 .explode('text')
 .group_by('text')
 .len()
 .sort('len', descending=True)
 )

text,len
str,u32
"""you""",268328
"""page""",251945
"""i""",146430
"""1""",141771
"""your""",136367
…,…
"""yashrobi""",1
"""meymai""",1
"""residenza""",1


### Most used word per document

In [138]:
some_file = sorted(glob.glob(DATA_DIR))[0]

df = (pl
      .read_parquet(some_file)
      # only HTML files (enum;2=html)
      .filter(pl.col("type") == 2)
      .filter(pl.col("lang") == "ENGLISH")
      # only important rows
      .select(pl.col("id").alias('doc_id'), pl.col("text").str.split(' ').alias("term"))
      .with_columns(doc_size=pl.col("term").list.len())
      .explode('term')
      .group_by('doc_id', 'doc_size', 'term')
      .len()
      .filter(pl.col("term").str.contains(r'\d|\.|"').not_())
      )

df

doc_id,doc_size,term,len
i64,u32,str,u32
810714821,906,"""pick""",1
810744200,1503,"""length""",3
810827051,4554,"""printer""",3
810672656,2367,"""接""",4
810710686,1834,"""muka""",2
…,…,…,…
810814444,374,"""capuliari""",1
810679495,1147,"""could""",9
810778512,887,"""eightfold""",3


In [139]:
term_count = (df
              .drop('doc_size')
              .sort('len', descending=True))

term_count

doc_id,term,len
i64,str,u32
810669784,"""amp""",46896
811049550,"""nigger""",42784
811050142,"""nigger""",34568
810660499,"""amp""",21029
811050135,"""nigger""",8223
…,…,…
810901339,"""normal""",1
810593072,"""modification""",1
810683923,"""personalize""",1


In [140]:
term_frequency = (df
                  .group_by('term')
                  .agg(pl.col("len").sum().alias('total_len'))
                  .join(term_count, left_on='term', right_on='term', )
                  .with_columns(tf=pl.col('len') / pl.col('total_len'))
                  .drop('len', 'total_len'))

term_frequency

term,doc_id,tf
str,i64,f64
"""amp""",810669784,0.612811
"""nigger""",811049550,0.499942
"""nigger""",811050142,0.403936
"""amp""",810660499,0.274795
"""nigger""",811050135,0.096088
…,…,…
"""normal""",810901339,0.0002
"""modification""",810593072,0.000902
"""personalize""",810683923,0.000633


In [141]:
# |C|
documents = df.shape[0]

inverse_document_frequency = (
    df
    .group_by('term')
    .agg(pl.col("doc_id").len().alias('doc_size'))
    .sort('doc_size', descending=True)
    .with_columns(idf=documents / pl.col("doc_size"))
    .with_columns(idf=pl.col("idf").log())
    .drop('doc_size')
    .sort('idf', descending=False)
)

inverse_document_frequency

term,idf
str,f64
"""you""",5.903366
"""page""",6.036316
"""use""",6.059673
"""privacy""",6.064843
"""all""",6.072762
…,…
"""tanium""",16.07206
"""localsupercell""",16.07206
"""deutlich""",16.07206


In [142]:
term_frequency_inverse_document_frequency = (
    term_frequency.join(
        inverse_document_frequency,
        left_on='term',
        right_on='term',
    )
    .with_columns(tf_idf=pl.col('tf') * pl.col('idf'))
    .drop('tf', 'idf')
    .sort('tf_idf', descending=True)
)

term_frequency_inverse_document_frequency

term,doc_id,tf_idf
str,i64,f64
"""ahojnne""",810964740,16.07206
"""boldsymbol""",810808441,16.07206
"""ccub""",810820530,16.07206
"""imigrasi""",810708878,16.07206
"""jüttler""",811018465,16.07206
…,…,…
"""you""",810834906,0.000022
"""you""",811127443,0.000022
"""you""",810939426,0.000022


In [144]:
term_frequency_inverse_document_frequency.filter(pl.col('doc_id') == 810964740)

term,doc_id,tf_idf
str,i64,f64
"""ahojnne""",810964740,16.07206
"""sarlinpe""",810964740,16.07206
"""whuaegeanse""",810964740,16.07206
"""anmatako""",810964740,16.07206
"""stonerl""",810964740,16.07206
…,…,…
"""any""",810964740,0.000135
"""out""",810964740,0.000114
"""site""",810964740,0.000106
